# 🌟 Aproximación de Estrellas en Campos 3D

Este notebook implementa aproximaciones de campos estelares 3D utilizando el método de **mínimos cuadrados**.

## Características:
- 📊 Regresión por mínimos cuadrados para ajuste de datos estelares
- 🔄 Funciones de loop para procesamiento iterativo
- 🪞 Funciones de reflexión para simetría espacial
- ⏳ Barra de progreso para monitoreo de cálculos
- 📈 Visualización 3D interactiva

In [ ]:
# Importación de librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm  # Barra de progreso para Jupyter
import pandas as pd
from scipy import linalg
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo para gráficos
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Clase Principal: StarField3D

Clase que maneja la generación y aproximación de campos estelares 3D.

In [ ]:
class StarField3D:
    """
    Clase para manejar campos estelares 3D con aproximación por mínimos cuadrados.
    
    Atributos:
        stars (np.array): Coordenadas 3D de las estrellas
        brightness (np.array): Brillo de cada estrella
        n_stars (int): Número de estrellas
    """
    
    def __init__(self, n_stars=100, seed=None):
        """
        Inicializa el campo estelar.
        
        Args:
            n_stars: Número de estrellas a generar
            seed: Semilla para reproducibilidad
        """
        if seed:
            np.random.seed(seed)
        
        self.n_stars = n_stars
        # Generar posiciones aleatorias en esfera unitaria
        self.stars = self._generate_sphere_points(n_stars)
        # Generar brillo aleatorio
        self.brightness = np.random.uniform(0.1, 1.0, n_stars)
        
    def _generate_sphere_points(self, n):
        """Genera puntos uniformemente distribuidos en una esfera."""
        phi = np.random.uniform(0, 2*np.pi, n)
        costheta = np.random.uniform(-1, 1, n)
        u = np.random.uniform(0, 1, n)
        
        theta = np.arccos(costheta)
        r = u**(1/3)
        
        x = r * np.sin(theta) * np.cos(phi)
        y = r * np.sin(theta) * np.sin(phi)
        z = r * np.cos(theta)
        
        return np.column_stack([x, y, z])
    
    def add_noise(self, noise_level=0.05):
        """Agrega ruido gaussiano a las posiciones."""
        noise = np.random.normal(0, noise_level, self.stars.shape)
        self.stars += noise
        return self
    
    def get_data(self):
        """Retorna las coordenadas y brillo."""
        return self.stars, self.brightness

## 2. Funciones de Mínimos Cuadrados

Implementación de aproximación por mínimos cuadrados para campos 3D.

In [ ]:
class LeastSquares3D:
    """
    Aproximación por mínimos cuadrados para datos 3D.
    Soporta polinomios de diferentes grados y funciones base.
    """
    
    def __init__(self, degree=2):
        self.degree = degree
        self.coefficients = None
        self.residuals = None
        
    def _build_design_matrix(self, X, Y):
        """
        Construye la matriz de diseño para el polinomio 2D.
        Incluye términos: 1, x, y, x², xy, y², ... hasta el grado especificado.
        """
        n_points = len(X)
        # Calcular número de términos: (degree+1)(degree+2)/2
        n_terms = (self.degree + 1) * (self.degree + 2) // 2
        A = np.zeros((n_points, n_terms))
        
        col = 0
        for i in range(self.degree + 1):
            for j in range(self.degree + 1 - i):
                A[:, col] = (X**i) * (Y**j)
                col += 1
        return A
    
    def fit(self, X, Y, Z, show_progress=True):
        """
        Ajusta el modelo por mínimos cuadrados: Z ≈ f(X,Y)
        
        Args:
            X, Y: Coordenadas independientes
            Z: Coordenada dependiente
            show_progress: Muestra barra de progreso
        """
        if show_progress:
            print("Construyendo matriz de diseño...")
            A = self._build_design_matrix(X, Y)
            
            print("Resolviendo sistema por mínimos cuadrados...")
            # Usar tqdm para mostrar progreso en cálculos pesados
            with tqdm(total=100, desc="Ajustando modelo") as pbar:
                # Resolver A * c = Z usando descomposición SVD
                U, s, Vh = np.linalg.svd(A, full_matrices=False)
                pbar.update(50)
                
                # Calcular coeficientes: c = V * diag(1/s) * U^T * Z
                s_inv = np.diag(1/s)
                self.coefficients = Vh.T @ s_inv @ U.T @ Z
                pbar.update(50)
        else:
            A = self._build_design_matrix(X, Y)
            self.coefficients, residuals, rank, s = np.linalg.lstsq(A, Z, rcond=None)
        
        # Calcular residuos
        Z_pred = self.predict(X, Y)
        self.residuals = Z - Z_pred
        
        return self
    
    def predict(self, X, Y):
        """Predice valores Z dados X e Y."""
        A = self._build_design_matrix(X, Y)
        return A @ self.coefficients
    
    def score(self, X, Y, Z):
        """Calcula R² del modelo."""
        Z_pred = self.predict(X, Y)
        ss_res = np.sum((Z - Z_pred)**2)
        ss_tot = np.sum((Z - np.mean(Z))**2)
        return 1 - (ss_res / ss_tot)

## 3. Funciones de Loop y Reflexión

Funciones para procesamiento iterativo y simetría espacial.

In [ ]:
def loop_star_analysis(star_field, degrees=[1, 2, 3, 4], noise_levels=[0.01, 0.05, 0.1]):
    """
    Ejecuta análisis en loop probando diferentes configuraciones.
    
    Args:
        star_field: Instancia de StarField3D
        degrees: Lista de grados polinómicos a probar
        noise_levels: Lista de niveles de ruido a probar
    
    Returns:
        DataFrame con resultados comparativos
    """
    results = []
    stars_original = star_field.stars.copy()
    
    total_iterations = len(degrees) * len(noise_levels)
    
    with tqdm(total=total_iterations, desc="Análisis en loop") as pbar:
        for degree in degrees:
            for noise in noise_levels:
                # Restaurar posiciones originales
                star_field.stars = stars_original.copy()
                
                # Agregar ruido
                star_field.add_noise(noise)
                X, Y, Z = star_field.stars[:, 0], star_field.stars[:, 1], star_field.stars[:, 2]
                
                # Ajustar modelo
                model = LeastSquares3D(degree=degree)
                model.fit(X, Y, Z, show_progress=False)
                
                # Guardar resultados
                results.append({
                    'degree': degree,
                    'noise_level': noise,
                    'r2_score': model.score(X, Y, Z),
                    'residual_std': np.std(model.residuals),
                    'n_coefficients': len(model.coefficients)
                })
                
                pbar.update(1)
    
    return pd.DataFrame(results)


def reflect_star_field(star_field, axis='z', plane_value=0):
    """
    Crea una reflexión del campo estelar respecto a un plano.
    
    Args:
        star_field: Instancia de StarField3D
        axis: Eje perpendicular al plano ('x', 'y', o 'z')
        plane_value: Valor donde se ubica el plano de reflexión
    
    Returns:
        Nueva instancia de StarField3D con estrellas reflejadas
    """
    axis_map = {'x': 0, 'y': 1, 'z': 2}
    axis_idx = axis_map.get(axis, 2)
    
    # Crear nuevo campo
    reflected = StarField3D(n_stars=star_field.n_stars * 2)
    
    # Copiar estrellas originales
    reflected.stars[:star_field.n_stars] = star_field.stars
    reflected.brightness[:star_field.n_stars] = star_field.brightness
    
    # Crear reflexión
    reflected_stars = star_field.stars.copy()
    reflected_stars[:, axis_idx] = 2 * plane_value - reflected_stars[:, axis_idx]
    
    # Agregar estrellas reflejadas
    reflected.stars[star_field.n_stars:] = reflected_stars
    reflected.brightness[star_field.n_stars:] = star_field.brightness * 0.8  # Reducir brillo
    
    return reflected


def create_symmetric_field(star_field, symmetry='mirror'):
    """
    Crea campos con diferentes tipos de simetría.
    
    Args:
        symmetry: 'mirror', 'rotational', 'dihedral'
    """
    if symmetry == 'mirror':
        # Reflexión simple en eje Z
        return reflect_star_field(star_field, 'z')
    
    elif symmetry == 'rotational':
        # Simetría rotacional de 180 grados
        rotated = StarField3D(n_stars=star_field.n_stars * 2)
        rotated.stars[:star_field.n_stars] = star_field.stars
        rotated.brightness[:star_field.n_stars] = star_field.brightness
        
        # Rotar 180 grados alrededor de Z
        rotated_stars = star_field.stars.copy()
        rotated_stars[:, 0] = -rotated_stars[:, 0]
        rotated_stars[:, 1] = -rotated_stars[:, 1]
        
        rotated.stars[star_field.n_stars:] = rotated_stars
        rotated.brightness[star_field.n_stars:] = star_field.brightness
        return rotated
    
    elif symmetry == 'dihedral':
        # Combinación de reflexiones
        temp = reflect_star_field(star_field, 'x')
        return reflect_star_field(temp, 'y')
    
    return star_field

## 4. Visualización 3D

Funciones para graficar resultados.

In [ ]:
def plot_star_field_3d(star_field, title="Campo Estelar 3D", figsize=(12, 9), show_surface=None):
    """
    Visualiza el campo estelar en 3D.
    
    Args:
        star_field: Instancia de StarField3D
        title: Título del gráfico
        figsize: Tamaño de la figura
        show_surface: Si se proporciona un modelo LeastSquares3D, muestra la superficie ajustada
    """
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    
    stars = star_field.stars
    brightness = star_field.brightness
    
    # Scatter plot de estrellas
    scatter = ax.scatter(stars[:, 0], stars[:, 1], stars[:, 2], 
                        c=brightness, cmap='plasma', s=brightness*100, 
                        alpha=0.8, edgecolors='white', linewidth=0.5)
    
    # Si se proporciona modelo, mostrar superficie
    if show_surface is not None:
        # Crear malla para superficie
        x_range = np.linspace(stars[:, 0].min(), stars[:, 0].max(), 30)
        y_range = np.linspace(stars[:, 1].min(), stars[:, 1].max(), 30)
        X_grid, Y_grid = np.meshgrid(x_range, y_range)
        
        # Predecir Z
        Z_pred = show_surface.predict(X_grid.flatten(), Y_grid.flatten())
        Z_grid = Z_pred.reshape(X_grid.shape)
        
        # Plot superficie
        ax.plot_surface(X_grid, Y_grid, Z_grid, alpha=0.3, cmap='viridis', 
                       linewidth=0, antialiased=True)
    
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('Z', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Colorbar para brillo
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.5, aspect=10)
    cbar.set_label('Brillo', rotation=270, labelpad=15)
    
    plt.tight_layout()
    plt.show()
    
    return fig


def plot_residuals_analysis(model, X, Y, Z):
    """
    Visualiza el análisis de residuos del modelo.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    Z_pred = model.predict(X, Y)
    residuals = Z - Z_pred
    
    # 1. Residuos vs Predicción
    axes[0, 0].scatter(Z_pred, residuals, alpha=0.6, c='blue', edgecolors='black')
    axes[0, 0].axhline(y=0, color='r', linestyle='--')
    axes[0, 0].set_xlabel('Valores Predichos')
    axes[0, 0].set_ylabel('Residuos')
    axes[0, 0].set_title('Residuos vs Predicción')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Histograma de residuos
    axes[0, 1].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    axes[0, 1].set_xlabel('Residuos')
    axes[0, 1].set_ylabel('Frecuencia')
    axes[0, 1].set_title('Distribución de Residuos')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Q-Q plot
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[1, 0])
    axes[1, 0].set_title('Q-Q Plot (Normalidad)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Residuos en espacio 3D
    ax3d = fig.add_subplot(224, projection='3d')
    scatter = ax3d.scatter(X, Y, Z, c=residuals, cmap='RdBu_r', s=50, alpha=0.8)
    ax3d.set_xlabel('X')
    ax3d.set_ylabel('Y')
    ax3d.set_zlabel('Z')
    ax3d.set_title('Residuos en Espacio 3D')
    plt.colorbar(scatter, ax=ax3d, shrink=0.5)
    
    plt.tight_layout()
    plt.show()
    
    return fig

## 5. Ejemplo de Uso Completo

Demostración de todas las funcionalidades.

In [ ]:
# Crear campo estelar
print("🌟 Generando campo estelar...")
field = StarField3D(n_stars=200, seed=42)

# Visualizar original
plot_star_field_3d(field, title="Campo Estelar Original")

In [ ]:
# Aplicar aproximación por mínimos cuadrados
print("\n📊 Aplicando mínimos cuadrados (grado 2)...")
X, Y, Z = field.stars[:, 0], field.stars[:, 1], field.stars[:, 2]

model = LeastSquares3D(degree=2)
model.fit(X, Y, Z, show_progress=True)

print(f"\n✅ R² del modelo: {model.score(X, Y, Z):.4f}")
print(f"📉 Desviación estándar de residuos: {np.std(model.residuals):.4f}")

In [ ]:
# Visualizar con superficie ajustada
plot_star_field_3d(field, title="Campo Estelar con Aproximación", show_surface=model)

In [ ]:
# Análisis de residuos
print("\n📈 Analizando residuos...")
plot_residuals_analysis(model, X, Y, Z)

In [ ]:
# Ejemplo de LOOP: Probar diferentes configuraciones
print("\n🔄 Ejecutando análisis en loop...")
results_df = loop_star_analysis(field, degrees=[1, 2, 3], noise_levels=[0.01, 0.05, 0.1])
print("\n📋 Resultados del análisis:")
display(results_df)

# Visualizar comparación
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² vs Grado para diferentes niveles de ruido
for noise in results_df['noise_level'].unique():
    subset = results_df[results_df['noise_level'] == noise]
    axes[0].plot(subset['degree'], subset['r2_score'], 'o-', 
                label=f'Ruido={noise}', linewidth=2, markersize=8)

axes[0].set_xlabel('Grado del Polinomio')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Precisión vs Complejidad del Modelo')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuos vs Ruido
for degree in results_df['degree'].unique():
    subset = results_df[results_df['degree'] == degree]
    axes[1].plot(subset['noise_level'], subset['residual_std'], 's-', 
                label=f'Grado={degree}', linewidth=2, markersize=8)

axes[1].set_xlabel('Nivel de Ruido')
axes[1].set_ylabel('Std de Residuos')
axes[1].set_title('Impacto del Ruido en la Aproximación')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Ejemplo de REFLEXIÓN: Crear simetrías
print("\n🪞 Creando campos con simetría...")

# Reflexión simple
field_reflected = reflect_star_field(field, axis='z')
plot_star_field_3d(field_reflected, title="Campo con Reflexión en Z")

# Simetría rotacional
field_rotational = create_symmetric_field(field, symmetry='rotational')
plot_star_field_3d(field_rotational, title="Campo con Simetría Rotacional")

# Simetría diedral
field_dihedral = create_symmetric_field(field, symmetry='dihedral')
plot_star_field_3d(field_dihedral, title="Campo con Simetría Diedral")

In [ ]:
# Comparación final de todos los campos
fig = plt.figure(figsize=(16, 12))

fields = [
    (field, "Original"),
    (field_reflected, "Reflejado"),
    (field_rotational, "Rotacional"),
    (field_dihedral, "Diedral")
]

for idx, (f, title) in enumerate(fields, 1):
    ax = fig.add_subplot(2, 2, idx, projection='3d')
    stars = f.stars
    brightness = f.brightness
    
    ax.scatter(stars[:, 0], stars[:, 1], stars[:, 2], 
              c=brightness, cmap='plasma', s=brightness*50, alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.suptitle('Comparación de Campos Estelares con Diferentes Simetrías', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n✨ ¡Análisis completo!")

## 6. Exportar Resultados

Guardar datos y modelos para uso posterior.

In [ ]:
# Funciones de exportación
def export_to_csv(star_field, filename='star_field_data.csv'):
    """Exporta coordenadas a CSV."""
    df = pd.DataFrame({
        'x': star_field.stars[:, 0],
        'y': star_field.stars[:, 1],
        'z': star_field.stars[:, 2],
        'brightness': star_field.brightness
    })
    df.to_csv(filename, index=False)
    print(f"✅ Datos exportados a {filename}")
    return df

def save_model_params(model, filename='model_params.npy'):
    """Guarda coeficientes del modelo."""
    np.save(filename, model.coefficients)
    print(f"✅ Parámetros guardados en {filename}")

# Ejemplo de uso
export_to_csv(field, 'mi_campo_estelar.csv')
save_model_params(model, 'coeficientes_modelo.npy')

print("\n📦 Archivos listos para subir a GitHub!")